In [ ]:
# Hugging face transformers, use pretrained model T5,
# Then we finetuned according to our need.
# USE small version -- T5 Small version
# COnditional generation projects ( MINI GEN AI project )

In [ ]:
#!pip install transformers

In [ ]:
#!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("/content/samsum-train.csv")
val_data = pd.read_csv("/content/samsum-validation.csv")


In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.sample(10)

,id,dialogue,summary
1615,13830115,"Lilly: Dearest Olivia, just talked about you w...",Bill had an emergency op in his eye due to a r...
12743,13680439,Mollie: where were you yesterday\r\nEthan: whe...,"Mollie suspects that Ethan is cheating on her,..."
4748,13809979,Pete: Wanna see my new baseball shirt?\r\nMike...,Pete sent Mike a picture of Pete's new basebal...
13449,13828143,Nettie: did ya take my headphones???\r\nTroy: ...,Troy took Nettie's headphones without asking a...
9220,13611827,Debbie: Do you have time-travel machines in Au...,Debbie can't use the clutch. Debbie wanted to ...
8544,13811408,Sam: You there yet?\r\nSonia: Yeah. Just arriv...,Sonia has just arrived at the party. There are...
6460,13728814,Graham: i'm getting a new computer and i know ...,Graham is getting a new computer. Jasper will ...
7692,13731071,Tom: Do you have Dr Ostrovsky’s phone number?\...,Mike gives Tom Dr Ostrovsky's number: 563-893-...
2778,13863237,Meghan: R u a True Blue? \nLily: Yep! I'm from...,Lily is a True Blue from Sydney. Meghan is fro...
8517,13821546,Malia: wanna join me for a snack?\r\nRichard: ...,Richard and Terry are joining Malia for a snac...


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

## Random sampling

In [ ]:
train_data = train_data.sample(n = 4000  , random_state = 42).reset_index(drop= True)
val_data = val_data.sample(n=500 , random_state= 42).reset_index(drop = True)

In [ ]:
train_data.shape

(4000, 3)

# Data Pre-processing

In [ ]:
import re
def clean_data(text):
  text = re.sub(r"\r\n", " ",text) # Lines
  text = re.sub(r"\s+", " ", text) # Spaces
  text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
  text = text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data['summary'].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [ ]:
tokenizer  = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512,truncation=True)
  targets = tokenizer(data["summary"], padding = "max_length", max_length=150 , truncation=True)

  inputs["labels"] = targets["input_ids"]
  return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize , axis = 1).tolist()

In [ ]:
#pip install --upgrade transformers

In [ ]:
print(type(tokenizer))

<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>


In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input_ids --> Toekon of Dialouge
# 1 --> End of sequence
# labels ---? Token of summary
# attebtion_mask ----> if val 1 => Valid data , if 0 => non valid data


In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
len(val_dataset[0]["labels"])

150

In [ ]:
type(train_dataset)

list

# Working with our model

In [ ]:
# NLP ==> generation taks
model = T5ForConditionalGeneration.from_pretrained("t5-small")

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch
if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
print("Device ", device)
model.to(device)

Device  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

# Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 7,
    weight_decay  = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500

)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.577381,0.380097
2,0.397476,0.359728
3,0.374439,0.354454
4,0.361381,0.349852
5,0.354290,0.348302
6,0.347821,0.347482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# 1. Model Load
# 2 . Odel Finetuned
# 3. Save the model.

# Saving Model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

# Test core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.


In [ ]:
import shutil

shutil.make_archive("saved_summary_model", 'zip', "saved_summary_model")

In [ ]:
import shutil
shutil.copytree("saved_summary_model", "/content/drive/MyDrive/saved_summary_model")

In [ ]:
from google.colab import files
files.download("saved_summary_model.zip")